# 00 · Data-Ready Pipeline

**First notebook in the Bee-A-Hero pipeline.** Runs top-to-bottom with no manual
intervention and leaves the iNaturalist dataset clean, balanced (70/15/15),
relabeled, and verified.

Stages: **inspect → validate → backup-verify → balance → relabel →
missing/duplicate/corrupted checks → directory consistency → final report.**

All heavy logic lives in `src/data_pipeline/` (`inaturalist_prep`, `label_tools`,
`eda`); this notebook only orchestrates and reports. Re-running is safe
(idempotent): the balancing step never re-deletes and never moves twice.

In [1]:
# --- Environment: locate repo root, make `src` importable ---
import sys, os, json
from pathlib import Path

here = Path.cwd()
for cand in [here, *here.parents]:
    if (cand / "src" / "config.py").exists():
        os.chdir(cand); sys.path.insert(0, str(cand)); break
REPO = Path.cwd()

from src import config as C
from src.data_pipeline import inaturalist_prep as prep
from src.data_pipeline import label_tools as labels
from src.data_pipeline import eda

REPORT = {}                                   # accumulates the final report
print("repo root :", REPO)
print("seed      :", C.SEED, "| ratios:", C.SPLIT_RATIOS)
print("target    :", C.TARGET_CLASS, "| labeled splits:", C.INAT_LABELED_SPLITS)

repo root : /home/manheim666/Desktop/Bee-A-Hero
seed      : 42 | ratios: {'train': 0.7, 'val': 0.15, 'test': 0.15}
target    : Insecta | labeled splits: ('train_mini', 'val')


## 1 · Dataset inspection

In [2]:
def inspect():
    info = {}
    for split in C.INAT_LABELED_SPLITS:
        d = C.INAT_DIR / split
        folders = [f for f in d.iterdir() if f.is_dir()]
        insecta = [f for f in folders if "_Insecta_" in f.name]
        imgs = sum(1 for _ in d.rglob("*") if _.is_file() and _.suffix.lower() in C.IMAGE_EXTS)
        info[split] = {"folders": len(folders), "insecta_folders": len(insecta),
                       "non_insecta_folders": len(folders) - len(insecta), "images": imgs}
    pt = C.INAT_DIR / C.INAT_UNLABELED_SPLIT
    info[C.INAT_UNLABELED_SPLIT] = {"images": sum(1 for _ in pt.glob('*') if _.is_file()),
                                    "note": "unlabeled — inference only, not in split"}
    return info

REPORT["inspection"] = inspect()
print(json.dumps(REPORT["inspection"], indent=2))

{
  "train_mini": {
    "folders": 2526,
    "insecta_folders": 2526,
    "non_insecta_folders": 0,
    "images": 126290
  },
  "val": {
    "folders": 2526,
    "insecta_folders": 2526,
    "non_insecta_folders": 0,
    "images": 25255
  },
  "public_test": {
    "images": 500000,
    "note": "unlabeled \u2014 inference only, not in split"
  }
}


## 2 · Backup verification
Confirm the Phase-3 safety backup exists and the original labels are intact (md5) **before** any mutating step.

In [3]:
import hashlib, subprocess
bk = C.BACKUP_DIR
checks = {"backup_dir_exists": bk.exists(),
          "original_labels_exist": (bk / "original_labels").is_dir(),
          "removed_bucket_exists": (bk / "removed").is_dir()}
md5_ok = True
sums = bk / "original_labels" / "MD5SUMS.txt"
if sums.exists():
    for line in sums.read_text().splitlines():
        want, name = line.split()
        p = bk / "original_labels" / name
        got = hashlib.md5(p.read_bytes()).hexdigest() if p.exists() else None
        md5_ok &= (got == want)
checks["original_labels_md5_ok"] = bool(md5_ok and sums.exists())
REPORT["backup_verification"] = checks
assert all(checks.values()), f"Backup verification failed: {checks}"
print(json.dumps(checks, indent=2)); print("Backup OK.")

{
  "backup_dir_exists": true,
  "original_labels_exist": true,
  "removed_bucket_exists": true,
  "original_labels_md5_ok": true
}
Backup OK.


## 3 · Dataset balancing (Insecta filter · exact dedup · 70/15/15)
Idempotent: non-Insecta and exact duplicates are *moved* to `data/_backup/removed/` (never deleted) and logged.

In [4]:
REPORT["balancing"] = prep.run(apply=True)     # safe to re-run
print(json.dumps(REPORT["balancing"], indent=2))

{
  "generated": "2026-07-01T07:32:44+00:00",
  "seed": 42,
  "ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "num_classes": 2526,
  "kept_images": 151545,
  "split_counts": {
    "train": 106077,
    "val": 22734,
    "test": 22734
  },
  "split_fractions": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "bee_images_per_split": {
    "train": 2604,
    "val": 558,
    "test": 558
  },
  "removed_total": 0,
  "removed_by_reason": {},
  "mode": "APPLIED"
}


## 4 · Label regeneration / update
Rebuild labels to reference only surviving images; filter the original iNat JSONs and emit clean per-split COCO labels.

In [5]:
res = labels.run()
REPORT["label_regeneration"] = {"filter_original": res["filter_original"],
                                "clean_labels": res["clean_labels"]}
REPORT["label_validation"] = res["validation"]
print(json.dumps(REPORT["label_validation"], indent=2))

{
  "num_classes": 2526,
  "manifest_images": 151545,
  "disk_images": 151545,
  "images_without_labels": 0,
  "labels_without_images": 0,
  "duplicated_labels": 0,
  "invalid_category_ids": 0,
  "corrupted_labels": 0,
  "cross_split_leakage": 0,
  "examples": {
    "images_without_labels": [],
    "labels_without_images": []
  },
  "all_checks_pass": true
}


## 5 · Integrity checks
Missing files · duplicate labels · corrupted images · near-duplicates · directory consistency.

In [6]:
df = eda.load_manifest_df()
paths = df["path"].tolist()

# 5a — missing files (manifest rows whose image is gone)
missing = [p for p in paths if not (REPO / p).exists()]

# 5b — duplicate labels (same image path listed twice)
dups = df["path"].value_counts()
duplicate_labels = dups[dups > 1].index.tolist()

# 5c — corrupted / unreadable images (FULL scan, all splits)
print(f"Scanning {len(paths):,} images for corruption ...")
corrupted = eda.scan_corrupted(paths, workers=os.cpu_count())

# 5d — perceptual near-duplicates (sampled)
near_dup_groups = eda.find_near_duplicates(paths, sample=8000)

REPORT["integrity"] = {
    "missing_files": len(missing),
    "duplicate_labels": len(duplicate_labels),
    "corrupted_images": len(corrupted),
    "near_duplicate_groups_sampled": len(near_dup_groups),
    "examples": {"missing": missing[:5],
                 "corrupted": [c[0] for c in corrupted[:5]]},
}
print(json.dumps(REPORT["integrity"], indent=2))

Scanning 151,545 images for corruption ...


{
  "missing_files": 0,
  "duplicate_labels": 0,
  "corrupted_images": 0,
  "near_duplicate_groups_sampled": 0,
  "examples": {
    "missing": [],
    "corrupted": []
  }
}


### 5e · Directory consistency

In [7]:
train_classes = {f.name for f in (C.INAT_DIR / "train_mini").iterdir() if f.is_dir()}
val_classes   = {f.name for f in (C.INAT_DIR / "val").iterdir() if f.is_dir()}
dir_checks = {
    "train_folders": len(train_classes),
    "val_folders": len(val_classes),
    "class_sets_identical": train_classes == val_classes,
    "manifest_classes": int(df["class_id"].nunique()),
    "all_paths_under_labeled_splits":
        bool(df["path"].str.startswith("data/raw/iNaturist/").all()),
    "every_class_has_all_splits":
        bool((df.pivot_table(index="class_name", columns="split",
              values="path", aggfunc="count", fill_value=0) > 0).all().all()),
}
REPORT["directory_consistency"] = dir_checks
print(json.dumps(dir_checks, indent=2))

{
  "train_folders": 2526,
  "val_folders": 2526,
  "class_sets_identical": true,
  "manifest_classes": 2526,
  "all_paths_under_labeled_splits": true,
  "every_class_has_all_splits": true
}


## 6 · Final report
Aggregate every check into a single PASS/FAIL gate and persist it. The notebook **fails loudly** (assert) if the dataset is not ready.

In [8]:
integ = REPORT["integrity"]; val = REPORT["label_validation"]; dc = REPORT["directory_consistency"]
gate = {
    "backup_verified": all(REPORT["backup_verification"].values()),
    "labels_valid": val["all_checks_pass"],
    "no_missing_files": integ["missing_files"] == 0,
    "no_duplicate_labels": integ["duplicate_labels"] == 0,
    "no_corrupted_images": integ["corrupted_images"] == 0,
    "class_sets_identical": dc["class_sets_identical"],
    "every_class_all_splits": dc["every_class_has_all_splits"],
    "split_is_70_15_15": REPORT["balancing"]["split_fractions"] ==
                         {"train": 0.7, "val": 0.15, "test": 0.15},
}
REPORT["gate"] = gate
REPORT["DATA_READY"] = all(gate.values())

out_dir = C.INTERIM_DIR / "reports"; out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "data_ready_report.json").write_text(json.dumps(REPORT, indent=2))

print(json.dumps(gate, indent=2))
print("\n=== DATA_READY:", REPORT["DATA_READY"], "===")
print("report ->", out_dir / "data_ready_report.json")
assert REPORT["DATA_READY"], "Dataset is NOT ready — see failing gate checks above."

{
  "backup_verified": true,
  "labels_valid": true,
  "no_missing_files": true,
  "no_duplicate_labels": true,
  "no_corrupted_images": true,
  "class_sets_identical": true,
  "every_class_all_splits": true,
  "split_is_70_15_15": true
}

=== DATA_READY: True ===
report -> /home/manheim666/Desktop/Bee-A-Hero/data/interim/reports/data_ready_report.json
